<a href="https://colab.research.google.com/github/mnkhaldi-MD/clinical-llm-safety-audit/blob/main/clinical_llm_safety_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import hashlib

# Configure environment settings
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

N_CASES = 50

# Construct the baseline evaluation cohort
df = pd.DataFrame({
    "case_id": [f"PT-{i:03d}" for i in range(1, N_CASES + 1)],
    "true_acuity": ["Routine"] * 42 + ["Urgent_RedFlag"] * 8,
    "true_glucose_value": np.round(np.random.uniform(5.2, 14.5, N_CASES), 1),
    "true_glucose_unit": "mmol/L",
    "true_egfr_cutoff": 45,
    "true_eligible": [True] * 35 + [False] * 15
})

# Inject baseline model (v1.0) predictions & known edge-case failures
df["v1_0_pred_acuity"] = df["true_acuity"]
df.loc[df["case_id"].isin(["PT-045", "PT-049"]), "v1_0_pred_acuity"] = "Routine"

df["v1_0_extracted_unit"] = "mmol/L"
df.loc[df["case_id"].isin(["PT-012", "PT-028"]), "v1_0_extracted_unit"] = "mg/dL"

df["v1_0_extracted_egfr"] = 45
df.loc[df["case_id"] == "PT-033", "v1_0_extracted_egfr"] = 60

# Model update (v1.1) regressions and drift behavior
df["v1_1_pred_acuity"] = df["v1_0_pred_acuity"]
df.loc[df["case_id"] == "PT-047", "v1_1_pred_acuity"] = "Routine"

df["v1_1_extracted_unit"] = df["v1_0_extracted_unit"]
df.loc[df["case_id"].isin(["PT-007", "PT-019"]), "v1_1_extracted_unit"] = "mg/dL"

df["v1_1_extracted_egfr"] = df["v1_0_extracted_egfr"]

print(f"Loaded {len(df)} clinical audit cases.")
print(f"Ground-truth urgent cases: {df['true_acuity'].eq('Urgent_RedFlag').sum()}")
display(df.head())

Loaded 50 clinical audit cases.
Ground-truth urgent cases: 8


,case_id,true_acuity,true_glucose_value,true_glucose_unit,true_egfr_cutoff,true_eligible,v1_0_pred_acuity,v1_0_extracted_unit,v1_0_extracted_egfr,v1_1_pred_acuity,v1_1_extracted_unit,v1_1_extracted_egfr
0,PT-001,Routine,8.7,mmol/L,45,True,Routine,mmol/L,45,Routine,mmol/L,45
1,PT-002,Routine,14.0,mmol/L,45,True,Routine,mmol/L,45,Routine,mmol/L,45
2,PT-003,Routine,12.0,mmol/L,45,True,Routine,mmol/L,45,Routine,mmol/L,45
3,PT-004,Routine,10.8,mmol/L,45,True,Routine,mmol/L,45,Routine,mmol/L,45
4,PT-005,Routine,6.7,mmol/L,45,True,Routine,mmol/L,45,Routine,mmol/L,45


In [5]:
def run_clinical_evaluation(data: pd.DataFrame, prefix: str) -> dict:
    """Calculates model accuracy, safety vulnerabilities, and protocol drifts."""
    pred_col = f"{prefix}_pred_acuity"
    unit_col = f"{prefix}_extracted_unit"
    egfr_col = f"{prefix}_extracted_egfr"

    # Primary Accuracies & Matches
    triage_correct = data[pred_col] == data["true_acuity"]
    unit_correct = data[unit_col] == data["true_glucose_unit"]
    egfr_correct = data[egfr_col] == data["true_egfr_cutoff"]

    # Red-Flag Sensitivity
    is_urgent = data["true_acuity"] == "Urgent_RedFlag"
    urgent_total = is_urgent.sum()
    missed_red_flags = (data.loc[is_urgent, pred_col] != "Urgent_RedFlag").sum()
    sensitivity = ((urgent_total - missed_red_flags) / urgent_total * 100) if urgent_total > 0 else 100.0

    # System Drift Counts
    unit_drifts = (~unit_correct).sum()
    egfr_drifts = (~egfr_correct).sum()

    # Safety Gate status
    strict_passes = triage_correct & unit_correct & egfr_correct

    return {
        "version": prefix.replace("_", "."),
        "accuracy_pct": round(triage_correct.mean() * 100, 1),
        "missed_red_flags_count": int(missed_red_flags),
        "red_flag_sensitivity_pct": round(sensitivity, 1),
        "unit_drifts_count": int(unit_drifts),
        "egfr_drifts_count": int(egfr_drifts),
        "safety_pass_pct": round(strict_passes.mean() * 100, 1)
    }

summary_df = pd.DataFrame([
    run_clinical_evaluation(df, "v1_0"),
    run_clinical_evaluation(df, "v1_1"),
])

display(summary_df)

,version,accuracy_pct,missed_red_flags_count,red_flag_sensitivity_pct,unit_drifts_count,egfr_drifts_count,safety_pass_pct
0,v1.0,96.0,2,75.0,2,1,90.0
1,v1.1,94.0,3,62.5,4,1,84.0


In [6]:
from datetime import datetime, timezone

# Define explicit registry and safety parameters
SAFETY_THRESHOLDS = {
    "min_sensitivity_pct": 80.0,
    "max_allowed_unit_drifts": 2
}

def build_governance_manifest(version_id: str, checkpoint: str, prompt: str, ontology: str, metrics: dict) -> dict:
    """Generates a deployment manifest with automated freeze criteria."""
    payload = f"{version_id}|{checkpoint}|{prompt}|{ontology}".encode("utf-8")
    fingerprint = hashlib.sha256(payload).hexdigest()[:16]

    # Evaluate against safety constraints
    fails_sensitivity = metrics["red_flag_sensitivity_pct"] < SAFETY_THRESHOLDS["min_sensitivity_pct"]
    fails_unit_drift = metrics["unit_drifts_count"] > SAFETY_THRESHOLDS["max_allowed_unit_drifts"]

    is_frozen = fails_sensitivity or fails_unit_drift

    return {
        "timestamp": datetime.now(timezone.utc).isoformat(timespec='seconds'),
        "model_version": version_id,
        "base_checkpoint": checkpoint,
        "ontology": ontology,
        "digest_hash": fingerprint,
        "lifecycle_status": "FLAGGED / SEVERE DRIFT" if is_frozen else "ACTIVE"
    }

manifests = [
    build_governance_manifest(
        version_id="v1.0",
        checkpoint="clinical-llm-base-2026-06",
        prompt="Extract glucose in mmol/L, eGFR cutoff, and flag urgent red-flag symptoms.",
        ontology="MedDRA-29.0 / SNOMED-2026",
        metrics=summary_df.iloc[0].to_dict()
    ),
    build_governance_manifest(
        version_id="v1.1",
        checkpoint="clinical-llm-base-2026-10",
        prompt="Extract glucose, eGFR cutoff, and triage acuity concisely.",
        ontology="MedDRA-29.0 / SNOMED-2026",
        metrics=summary_df.iloc[1].to_dict()
    )
]

governance_registry = pd.DataFrame(manifests)
display(governance_registry)

,timestamp,model_version,base_checkpoint,ontology,digest_hash,lifecycle_status
0,2026-11-18T08:52:27+00:00,v1.0,clinical-llm-base-2026-06,MedDRA-29.0 / SNOMED-2026,c059fe7096a9fcbc,FLAGGED / SEVERE DRIFT
1,2026-11-18T08:52:27+00:00,v1.1,clinical-llm-base-2026-10,MedDRA-29.0 / SNOMED-2026,a99894dbd9f345ff,FLAGGED / SEVERE DRIFT
